# 08 — Modelos no supervisados: PCA y clustering

Objetivos:
- Construir un pipeline no supervisado con escalado, PCA y K-Means.
- Evaluar clusters con `silhouette_score`.
- Interpretar grupos sin usar el target como etiqueta de entrenamiento.
- Visualizar segmentos de partidos según cuotas.

In [ ]:
%load_ext kedro.ipython

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "raw" / "database.sqlite"
assert (PROJECT_ROOT / "pyproject.toml").is_file(), (
    "Abre este notebook desde la raíz del proyecto con `kedro jupyter lab`."
)
assert DB_PATH.is_file(), (
    "No existe data/raw/database.sqlite. Ejecuta: `python scripts/bootstrap_data.py`."
)
print(f"Proyecto: {PROJECT_ROOT}")
print(f"SQLite: {DB_PATH}")

## 1. Datos

En no supervisado no usamos `outcome` para entrenar. Lo dejamos solo para interpretar después si los clusters separan perfiles de partido.

In [ ]:
import numpy as np
import pandas as pd

match = catalog.load("match")
goal_diff = match["home_team_goal"] - match["away_team_goal"]
outcome = np.select([goal_diff > 0, goal_diff == 0, goal_diff < 0], [2, 1, 0])
feat_cols = ["B365H", "B365D", "B365A"]
df = match[feat_cols].assign(outcome=outcome).dropna().reset_index(drop=True)
X = df[feat_cols]
print(df.shape)
df.head()

## 2. PCA para visualizar

PCA reduce las tres cuotas a dos componentes. No “predice”; resume variabilidad para mirar patrones.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pca_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=2, random_state=42)),
])
components = pca_pipe.fit_transform(X)
pca_df = pd.DataFrame(components, columns=["PC1", "PC2"])
pca_df["outcome"] = df["outcome"].map({0: "visita", 1: "empate", 2: "local"})
print("Varianza explicada:", pca_pipe.named_steps["pca"].explained_variance_ratio_.round(3))
pca_df.head()

In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.figure(figsize=(6, 4))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="outcome", alpha=0.45, s=25)
plt.title("PCA de cuotas Bet365 (color solo para interpretar)")
plt.show()

## 3. K-Means y selección de k

Probamos varios valores de `k`. Silhouette cercano a 1 indica clusters más separados; cercano a 0 indica solapamiento.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

rows = []
X_scaled = StandardScaler().fit_transform(X)
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X_scaled)
    rows.append({"k": k, "silhouette": silhouette_score(X_scaled, labels)})

silhouette_df = pd.DataFrame(rows)
silhouette_df

In [ ]:
best_k = int(silhouette_df.sort_values("silhouette", ascending=False).iloc[0]["k"])
cluster_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("cluster", KMeans(n_clusters=best_k, random_state=42, n_init="auto")),
])
df_clusters = df.copy()
df_clusters["cluster"] = cluster_pipe.fit_predict(X)
print("k elegido:", best_k)
df_clusters.groupby("cluster")[feat_cols].mean().round(2)

## 4. Interpretación de clusters

Después de entrenar sin etiquetas, miramos cómo se distribuye el resultado real en cada grupo. Esto es interpretación, no supervisión.

In [ ]:
profile = pd.crosstab(
    df_clusters["cluster"],
    df_clusters["outcome"].map({0: "visita", 1: "empate", 2: "local"}),
    normalize="index",
).round(3)
profile

In [ ]:
plot_df = pca_df.copy()
plot_df["cluster"] = df_clusters["cluster"].astype(str)
plt.figure(figsize=(6, 4))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="cluster", alpha=0.55, s=25)
plt.title("Clusters K-Means vistos en el plano PCA")
plt.show()

## Cierre

Los modelos no supervisados sirven para explorar estructura, no para medir “accuracy”. La pregunta correcta es: ¿los grupos tienen sentido para el negocio o ayudan a formular mejores hipótesis?